In [0]:
#  Transformations: 
# 1. Standardize team names
# . Aggregate match results to team-season level
# 3. Calculate season statistics (wins, draws, losses, goals, points)
# 4. Validate data quality
#  Transform raw match-by-match data into clean team-season statistics

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Load bronze data
df_matches = spark.table("workspace.premier_league_bronze.pl_matches_historical")

# Select only columns we need
df_matches_clean = df_matches.select(
    "season",
    "Date",
    "HomeTeam",
    "AwayTeam", 
    "FTHG",  # Full Time Home Goals
    "FTAG",  # Full Time Away Goals
    "FTR"    # Full Time Result (H/A/D)
)

print(f"Total matches loaded: {df_matches_clean.count()}")
df_matches_clean.show(5)

Total matches loaded: 1900
+-------+----------+--------------+-------------+----+----+---+
| season|      Date|      HomeTeam|     AwayTeam|FTHG|FTAG|FTR|
+-------+----------+--------------+-------------+----+----+---+
|2022-23|21/01/2023|   Bournemouth|Nott'm Forest|   1|   1|  D|
|2022-23|21/01/2023|     Leicester|     Brighton|   2|   2|  D|
|2022-23|21/01/2023|   Southampton|  Aston Villa|   0|   1|  A|
|2022-23|21/01/2023|      West Ham|      Everton|   2|   0|  H|
|2022-23|21/01/2023|Crystal Palace|    Newcastle|   0|   0|  D|
+-------+----------+--------------+-------------+----+----+---+
only showing top 5 rows


In [0]:
# Create team name mapping for standardization
team_name_mapping = {
    "Man United": "Manchester United",
    "Man City": "Manchester City",
    "Spurs": "Tottenham",
    "Newcastle": "Newcastle United",
    "Nott'm Forest": "Nottingham Forest",
    "Leeds": "Leeds United",
    "Sheffield United": "Sheffield Utd",
    "West Ham": "West Ham United",
    "Brighton": "Brighton & Hove Albion",
    # Add more mappings as needed
}

# Apply mapping to both home and away teams
def standardize_name(team_name):
    return team_name_mapping.get(team_name, team_name)

standardize_udf = F.udf(standardize_name)

df_matches_clean = df_matches_clean.withColumn(
    "HomeTeam", standardize_udf(F.col("HomeTeam"))
).withColumn(
    "AwayTeam", standardize_udf(F.col("AwayTeam"))
)

In [0]:
# Calculate home statistics for each team
df_home_stats = df_matches_clean.groupBy("season", "HomeTeam").agg(
    F.count("*").alias("home_games"),
    F.sum(F.when(F.col("FTR") == "H", 1).otherwise(0)).alias("home_wins"),
    F.sum(F.when(F.col("FTR") == "D", 1).otherwise(0)).alias("home_draws"),
    F.sum(F.when(F.col("FTR") == "A", 1).otherwise(0)).alias("home_losses"),
    F.sum("FTHG").alias("home_goals_for"),
    F.sum("FTAG").alias("home_goals_against")
).withColumnRenamed("HomeTeam", "team")

print("Home statistics sample:")
df_home_stats.show(5)

Home statistics sample:
+-------+--------------------+----------+---------+----------+-----------+--------------+------------------+
| season|                team|home_games|home_wins|home_draws|home_losses|home_goals_for|home_goals_against|
+-------+--------------------+----------+---------+----------+-----------+--------------+------------------+
|2023-24|         Aston Villa|        19|       12|         4|          3|            48|                28|
|2024-25|           Liverpool|        19|       14|         4|          1|            42|                16|
|2024-25|           Leicester|        19|        4|         3|         12|            15|                34|
|2024-25|   Manchester United|        19|        7|         3|          9|            23|                28|
|2022-23|Brighton & Hove A...|        19|       10|         4|          5|            37|                21|
+-------+--------------------+----------+---------+----------+-----------+--------------+---------------

In [0]:
# Calculate away statistics for each team
df_away_stats = df_matches_clean.groupBy("season", "AwayTeam").agg(
    F.count("*").alias("away_games"),
    F.sum(F.when(F.col("FTR") == "A", 1).otherwise(0)).alias("away_wins"),
    F.sum(F.when(F.col("FTR") == "D", 1).otherwise(0)).alias("away_draws"),
    F.sum(F.when(F.col("FTR") == "H", 1).otherwise(0)).alias("away_losses"),
    F.sum("FTAG").alias("away_goals_for"),
    F.sum("FTHG").alias("away_goals_against")
).withColumnRenamed("AwayTeam", "team")

print("Away statistics sample:")
df_away_stats.show(5)

Away statistics sample:
+-------+--------------------+----------+---------+----------+-----------+--------------+------------------+
| season|                team|away_games|away_wins|away_draws|away_losses|away_goals_for|away_goals_against|
+-------+--------------------+----------+---------+----------+-----------+--------------+------------------+
|2023-24|         Aston Villa|        19|        8|         4|          7|            28|                33|
|2024-25|           Liverpool|        19|       11|         5|          3|            44|                25|
|2024-25|           Leicester|        19|        2|         4|         13|            18|                46|
|2024-25|   Manchester United|        19|        4|         6|          9|            21|                26|
|2022-23|Brighton & Hove A...|        19|        8|         4|          7|            35|                32|
+-------+--------------------+----------+---------+----------+-----------+--------------+---------------

In [0]:
# Join home and away stats
df_combined = df_home_stats.join(df_away_stats, on=["season", "team"], how="outer")

# Calculate total season statistics
df_season_stats = df_combined.withColumn(
    "games_played", F.col("home_games") + F.col("away_games")
).withColumn(
    "wins", F.col("home_wins") + F.col("away_wins")
).withColumn(
    "draws", F.col("home_draws") + F.col("away_draws")
).withColumn(
    "losses", F.col("home_losses") + F.col("away_losses")
).withColumn(
    "goals_for", F.col("home_goals_for") + F.col("away_goals_for")
).withColumn(
    "goals_against", F.col("home_goals_against") + F.col("away_goals_against")
).withColumn(
    "goal_difference", F.col("goals_for") - F.col("goals_against")
).withColumn(
    "points", (F.col("wins") * 3) + F.col("draws")
)

# Calculate position (rank by points, then goal difference)
window_spec = Window.partitionBy("season").orderBy(
    F.col("points").desc(), 
    F.col("goal_difference").desc(),
    F.col("goals_for").desc()
)

df_season_stats = df_season_stats.withColumn(
    "position", F.row_number().over(window_spec)
)

# Select final columns
df_final = df_season_stats.select(
    "season",
    "team",
    "position",
    "games_played",
    "wins",
    "draws", 
    "losses",
    "goals_for",
    "goals_against",
    "goal_difference",
    "points",
    "home_wins",
    "home_draws",
    "home_losses",
    "home_goals_for",
    "home_goals_against",
    "away_wins",
    "away_draws",
    "away_losses",
    "away_goals_for",
    "away_goals_against"
)

print(f"Total team-seasons: {df_final.count()}")
df_final.orderBy("season", "position").show(20)


Total team-seasons: 100
+-------+--------------------+--------+------------+----+-----+------+---------+-------------+---------------+------+---------+----------+-----------+--------------+------------------+---------+----------+-----------+--------------+------------------+
| season|                team|position|games_played|wins|draws|losses|goals_for|goals_against|goal_difference|points|home_wins|home_draws|home_losses|home_goals_for|home_goals_against|away_wins|away_draws|away_losses|away_goals_for|away_goals_against|
+-------+--------------------+--------+------------+----+-----+------+---------+-------------+---------------+------+---------+----------+-----------+--------------+------------------+---------+----------+-----------+--------------+------------------+
|2020-21|     Manchester City|       1|          38|  27|    5|     6|       83|           32|             51|    86|       13|         2|          4|            43|                17|       14|         3|          2|   

In [0]:
# Validation checks
print("=== Data Quality Checks ===\n")

# Check 1: Each season should have 20 teams
teams_per_season = df_final.groupBy("season").count().orderBy("season")
print("Teams per season:")
teams_per_season.show()

# Check 2: Each team should play 38 games
games_check = df_final.filter(F.col("games_played") != 38)
print(f"\nTeams with != 38 games: {games_check.count()}")
if games_check.count() > 0:
    games_check.select("season", "team", "games_played").show()

# Check 3: Points calculation should be correct
points_check = df_final.filter(
    F.col("points") != ((F.col("wins") * 3) + F.col("draws"))
)
print(f"\nTeams with incorrect points calculation: {points_check.count()}")

# Check 4: Games played = wins + draws + losses
games_math_check = df_final.filter(
    F.col("games_played") != (F.col("wins") + F.col("draws") + F.col("losses"))
)
print(f"\nTeams with incorrect games math: {games_math_check.count()}")

# Check 5: Display top teams from each season
print("\nChampions from each season:")
champions = df_final.filter(F.col("position") == 1).orderBy("season")
champions.select("season", "team", "points", "wins", "goal_difference").show()

print("\nValidation complete!")

=== Data Quality Checks ===

Teams per season:
+-------+-----+
| season|count|
+-------+-----+
|2020-21|   20|
|2021-22|   20|
|2022-23|   20|
|2023-24|   20|
|2024-25|   20|
+-------+-----+


Teams with != 38 games: 0

Teams with incorrect points calculation: 0

Teams with incorrect games math: 0

Champions from each season:
+-------+---------------+------+----+---------------+
| season|           team|points|wins|goal_difference|
+-------+---------------+------+----+---------------+
|2020-21|Manchester City|    86|  27|             51|
|2021-22|Manchester City|    93|  29|             73|
|2022-23|Manchester City|    89|  28|             61|
|2023-24|Manchester City|    91|  28|             62|
|2024-25|      Liverpool|    84|  25|             45|
+-------+---------------+------+----+---------------+


Validation complete!


In [0]:
# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.premier_league_silver")

# Save as Delta table
df_final.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.premier_league_silver.team_season_stats"
)

print(" Silver table created: workspace.premier_league_silver.team_season_stats")
print(f"   Total records: {df_final.count()}")


 Silver table created: workspace.premier_league_silver.team_season_stats
   Total records: 100
